# Creator S7 · Render a Short in Colab

Open this notebook in Chrome on your phone or computer. In Colab, choose **Runtime → Change runtime type → T4 GPU** if it is offered, then connect. GPU availability is controlled by Google and may vary.

**First setup downloads about 0.9 GB** of voice/alignment models plus Python packages. Use Wi-Fi. The notebook does not start that download until you turn `RUN_FULL_SETUP` on in the setup cell.

This notebook renders the included **Inside Out / swallow-gum demo** with the original Faceless Shorts Chrome, Kokoro, Whisper and FFmpeg workflow. It saves the MP4 to your Google Drive. A free Colab session can disconnect, and the GPU is not guaranteed.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess
drive.mount('/content/drive')
REPO = Path('/content/Creator-S7')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SparrowO7/Creator-S7.git', str(REPO)], check=True)
SKILL = REPO / 'skills/faceless-shorts'
os.environ['FACELESS_HOME'] = '/content/faceless-shorts'
chrome = next((p for p in ['/usr/bin/google-chrome','/usr/bin/google-chrome-stable','/usr/bin/chromium','/usr/bin/chromium-browser'] if Path(p).exists()), None)
if chrome: os.environ['CHROME'] = chrome
for cmd in [['node','--version'],['ffmpeg','-version'],['nvidia-smi']]:
    print('\n$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True); print((r.stdout or r.stderr).splitlines()[0] if (r.stdout or r.stderr) else 'not available')
print('Chrome:', chrome or 'not found yet; setup will report if missing')
print('Skill source:', SKILL)

## 1 · One-time model setup

Turn `RUN_FULL_SETUP` to `True` only when you are ready for the model download. Run this cell and wait for **All set**. Keep the same Colab session open for the render cells below.

In [ ]:
RUN_FULL_SETUP = False  # change to True on Wi-Fi for the ~0.9 GB first-time setup
if not RUN_FULL_SETUP:
    print('Setup skipped to protect your data. Set RUN_FULL_SETUP = True on Wi-Fi, then run this cell again.')
else:
    subprocess.run(['bash', str(SKILL / 'scripts/setup.sh')], cwd=REPO, env=os.environ.copy(), check=True)

## 2 · Render the sample

Run this after setup prints **All set**. It creates the sample narration, aligns word captions, checks the scene, makes the contact sheet, renders the MP4 and copies it into `My Drive/CreatorS7/`.

In [ ]:
import subprocess
PROJECTS = Path('/content/faceless-shorts/videos')
STYLE = 'inside-out'  # try: inside-out, night-file, deep-dive, stick-story, paper-atlas, tall-tale
PROJECT = PROJECTS / f'{STYLE}-colab-demo'
def run(args, cwd=REPO):
    print('$', ' '.join(map(str,args)))
    subprocess.run(list(map(str,args)), cwd=cwd, env=os.environ.copy(), check=True)
if not (Path(os.environ['FACELESS_HOME'])/'kokoro/kokoro-v1.0.onnx').exists():
    raise RuntimeError('Run the model setup cell first. Set RUN_FULL_SETUP=True there, on Wi-Fi.')
if PROJECT.exists(): shutil.rmtree(PROJECT)
run(['bash', SKILL/'scripts/new_video.sh', STYLE, PROJECT.name, '--dir', PROJECTS])
run([SKILL/'scripts/py', SKILL/'scripts/voice.py', PROJECT])
run([SKILL/'scripts/py', SKILL/'scripts/words.py', PROJECT])
run(['node','tools/capture.mjs','check'], cwd=PROJECT)
run(['node','tools/capture.mjs','cues'], cwd=PROJECT)
run([SKILL/'scripts/py','tools/mix.py','.'], cwd=PROJECT)
run(['node','tools/capture.mjs','sheet','review/all.jpg','1','8'], cwd=PROJECT)
from IPython.display import display, Image
display(Image(filename=str(PROJECT/'review/all.jpg')))

## 3 · Export and save the MP4

Look over the contact sheet above. If it looks okay, run this cell. The sample is 30 seconds; 3D styles can take much longer, especially when Chrome falls back to software rendering.

In [ ]:
run(['node','tools/capture.mjs','video','renders/creator-s7-demo.mp4'], cwd=PROJECT)
EXPORTS = Path('/content/drive/MyDrive/CreatorS7')
EXPORTS.mkdir(parents=True, exist_ok=True)
out = EXPORTS / 'creator-s7-demo.mp4'
shutil.copy2(PROJECT/'renders/creator-s7-demo.mp4', out)
print('Saved to Google Drive:', out)
print('Open Google Drive on your phone → My Drive → CreatorS7 → creator-s7-demo.mp4')